In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from timm.layers import trunc_normal_ 

In [ ]:
# 1. Cấu hình
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 102  # Flowers102 có 102 lớp

# 2. Transform dữ liệu
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Tải dữ liệu (Flowers102)
train_dataset = datasets.Flowers102(root='./data', split='train', download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)



In [ ]:
# 4. Định nghĩa Model DeiT
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_ch=3, embed_dim=192):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x) 
        return x.flatten(2).transpose(1, 2) 

In [ ]:
class DeiT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=102, 
                 embed_dim=192, depth=12, heads=3, 
                 dropout=0.1, mlp_ratio=4.0): # <-- Thêm tham số mới
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        num_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 2, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout) # Dropout cho Position Embedding

        # Tùy chỉnh lớp Transformer với dim_feedforward và dropout
        hidden_dim = int(embed_dim * mlp_ratio)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=heads, 
            dim_feedforward=hidden_dim, # Độ rộng lớp ẩn
            dropout=dropout,            # Tỉ lệ dropout
            batch_first=True
        )
        self.blocks = nn.TransformerEncoder(enc_layer, num_layers=depth)

        self.head = nn.Linear(embed_dim, num_classes)
        self.head_dist = nn.Linear(embed_dim, num_classes)

        trunc_normal_(self.pos_embed, std=.02)
        trunc_normal_(self.cls_token, std=.02)
        trunc_normal_(self.dist_token, std=.02)

    def forward(self, x):
        b = x.shape[0]
        x = self.patch_embed(x)
        cls_token = self.cls_token.expand(b, -1, -1)
        dist_token = self.dist_token.expand(b, -1, -1)
        
        x = torch.cat((cls_token, dist_token, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x) # Áp dụng dropout
        
        x = self.blocks(x)
        return self.head(x[:, 0]), self.head_dist(x[:, 1])


In [ ]:
teacher = models.resnet50(weights='IMAGENET1K_V1').to(device)
teacher.fc = nn.Linear(teacher.fc.in_features, NUM_CLASSES).to(device)
teacher.eval() 

student = DeiT(
    num_classes=102, 
    embed_dim=256, # Tăng nhẹ độ rộng
    depth=8,       # Giảm độ sâu để chạy nhanh hơn nếu cần
    heads=4, 
    dropout=0.15,  # Tăng khả năng chống overfitting
    mlp_ratio=3.0  # Tỉ lệ lớp ẩn
).to(device)
# Thêm Weight Decay vào Optimizer (thông số 0.05 là tiêu chuẩn cho ViT/DeiT)
optimizer = optim.AdamW(student.parameters(), lr=1e-3, weight_decay=0.05)
# Thêm Scheduler: Giảm LR sau mỗi 5 epoch để hội tụ tốt hơn
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
criterion = nn.CrossEntropyLoss()
distill_criterion = nn.KLDivLoss(reduction='batchmean')

# 7. Hàm huấn luyện đã sửa lỗi tham số
def train_one_epoch(epoch, T=3.0, alpha=0.5):
    student.train()
    total_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        with torch.no_grad():
            teacher_logits = teacher(images)
            
        cls_logits, dist_logits = student(images)
        
        # Base loss (nhãn thật)
        base_loss = criterion(cls_logits, labels)
        
        # Distillation loss (Teacher knowledge)
        soft_distill_loss = distill_criterion(
            torch.log_softmax(dist_logits / T, dim=1),
            torch.softmax(teacher_logits / T, dim=1)
        ) * (T * T)
        
        loss = (1 - alpha) * base_loss + alpha * soft_distill_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
for epoch in range(10):
    train_one_epoch(epoch)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

def visualize_prediction():
    student.eval()  # Chuyển mô hình sang chế độ dự báo
    
    # 1. Lấy một ảnh ngẫu nhiên từ dataset
    idx = random.randint(0, len(train_dataset) - 1)
    image, label = train_dataset[idx]
    
    # 2. Xử lý ảnh để đưa vào mô hình
    # Thêm dimension batch (1, C, H, W) và đưa lên device
    input_tensor = image.unsqueeze(0).to(device)
    
    # 3. Dự đoán
    with torch.no_grad():
        # DeiT trả về 2 output: cls_logits và dist_logits
        output_cls, _ = student(input_tensor)
        prediction = torch.argmax(output_cls, dim=1).item()
    
    # 4. Chuyển ảnh từ Tensor về dạng hiển thị được (H, W, C)
    # Cần đảo ngược quá trình Normalize để màu sắc hiển thị đúng
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    img_display = inv_normalize(image).permute(1, 2, 0).cpu().numpy()
    img_display = np.clip(img_display, 0, 1) # Đảm bảo giá trị pixel trong khoảng [0, 1]

    # 5. Hiển thị
    plt.figure(figsize=(6, 6))
    plt.imshow(img_display)
    title_color = 'green' if prediction == label else 'red'
    plt.title(f"Nhãn thật: {label} \n Dự đoán: {prediction}", color=title_color, fontsize=14)
    plt.axis('off')
    plt.show()

# Gọi hàm để xem ví dụ
visualize_prediction()
